# M14 on Kaggle: E1–E4 and E7 at one seed, with per-client counts

**Set two things in the first code cell:** `SEED` (42, 43 or 44) and `PART` ("A" or "B").

- **Part A** (about 8 h): M3 + M4 (E1, static), M6 (naive and local incremental), M7 (E3, FedMD), M8 (FedAvg, FedProx, lr 0.03).
- **Part B** (about 10 h): M9 (E4, all passes) + the two M14 ablations of E4, then M13 (E7).

Part B needs Part A's results of the same seed in the repository, so run **A first, bring its zip back, and run B after it is committed**. Different seeds can run at the same time.

Seed 42 rewrites the main reports (`reports/mX/full/`), seeds 43 and 44 write to `seed43/` and `seed44/`. The partition is the same for every seed.

**Attach as input:** the private Kaggle dataset with the M1 `processed/` files, as for every earlier notebook.
**Settings:** Internet *On*. Accelerator: *None*.
**Run:** *Save Version → Save & Run All (Commit)*, so it keeps running with the browser closed.

Each step runs in its own process and the zip is rewritten after each one, so a failure loses nothing that finished. `exit code: 137` means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m14_seed<SEED>_part<PART>_outputs.zip`. Download it from the *Output* tab and bring it back, also after a failure.


In [ ]:
SEED = 42      # 42, 43 or 44
PART = "A"     # "A" first, then "B" once A's zip has been committed
TAG = "" if SEED == 42 else f"seed{SEED}"
TAG_ARG = f"--tag {TAG}" if TAG else ""
SET = f"--set resources.n_jobs=4 seed={SEED}"   # 4 threads: results reproduce exactly only at the same thread count
import glob, os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert SEED in (42, 43, 44) and PART in ("A", "B"), (SEED, PART)
assert os.path.exists("src/ids/eval/stats.py"), "This clone predates M14. Restart the session so it clones the latest main."
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "The processed dataset is not attached: Add Input -> your M1 processed dataset."
if PART == "B":
    m7 = os.path.join("reports/m7/full", TAG, "summary.json")
    assert os.path.exists(m7) and "csv_client" in open(os.path.join("reports/m7/full", TAG, "confusions.csv")).read(), \
        f"Part A of seed {SEED} isn't in the repository yet ({m7} with per-client counts). Run part A and bring its zip back first."


In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'


In [ ]:
import shutil
MS = ["m3", "m4", "m6", "m7", "m8"] if PART == "A" else ["m9", "m13"]
OUT = f"/kaggle/working/m14_seed{SEED}_part{PART}_outputs"
def pack():
    stage = "/kaggle/working/m14_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    for m in MS:
        rep = os.path.join("reports", m, "full", TAG)
        if os.path.isdir(rep):
            # seed 42's folder holds the other seeds' folders too; only this seed's files go in the zip
            skip = shutil.ignore_patterns("checkpoints", "seed4*", "lr0.1", "mistral") if not TAG else shutil.ignore_patterns("checkpoints")
            shutil.copytree(rep, f"{stage}/{rep}", ignore=skip)
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_full")):
        if any(f"_{m}_" in os.path.basename(r) for m in MS):
            shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}", ignore=shutil.ignore_patterns("models"))
    shutil.make_archive(OUT, "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", OUT + ".zip")

def step(cmd):
    print("=====", cmd, flush=True)
    rc = subprocess.run(f"python -X faulthandler {cmd}", shell=True).returncode
    print("exit code:", rc, flush=True)
    pack()
    return rc


In [ ]:
if PART == "A":
    if step(f"scripts/m3_baselines.py --env kaggle --pool full {SET} {TAG_ARG}") == 0:
        step(f"scripts/m4_evaluate.py --env kaggle --pool full {SET} {TAG_ARG}")   # the M3 run just written is the latest
    step(f"scripts/m6_incremental.py --env kaggle --pool full {SET} {TAG_ARG} --resume")
    for name in ["e3", "fedmd"]:
        step(f"scripts/m7_distill.py --env kaggle --pool full {SET} {TAG_ARG} --only {name} --resume")
    step(f"scripts/m7_distill.py --env kaggle --pool full {SET} {TAG_ARG} --report")
    for name in ["fedavg", "fedavg_no_rehearsal", "fedprox_mu0.001", "fedprox_mu0.01", "fedprox_mu0.1", "fedprox_mu1"]:
        step(f"scripts/m8_fedavg.py --env kaggle --pool full {SET} fedavg.lr=0.03 {TAG_ARG} --only {name} --resume")
    step(f"scripts/m8_fedavg.py --env kaggle --pool full {SET} fedavg.lr=0.03 {TAG_ARG} --report")


In [ ]:
if PART == "B":
    for name in ["fixed", "fixed_k10", "fixed_k20", "fixed_k30", "e4_schedule", "e4_label_free", "e4", "e4_added"]:
        step(f"scripts/m9_e4.py --env kaggle --pool full {SET} {TAG_ARG} --only {name} --resume")
    # the ablations before the report, while the K sweep's checkpoints still exist
    for name in ["e4_added_allfeat", "e4_added_noreh"]:
        step(f"scripts/m9_e4.py --env kaggle --pool full {SET} {TAG_ARG} --ablation --only {name} --resume")
    step(f"scripts/m9_e4.py --env kaggle --pool full {SET} {TAG_ARG} --ablation --report")
    step(f"scripts/m9_e4.py --env kaggle --pool full {SET} {TAG_ARG} --report")
    step(f"scripts/m13_fedgbdt.py --env kaggle --pool full {SET} {TAG_ARG}")


In [ ]:
# What finished, and whether every run has per-client counts
for m in MS:
    f = os.path.join("reports", m, "full", TAG, "confusions.csv")
    ok = os.path.exists(f) and "csv_client" in open(f).read()
    print(f"{m}: {'per-client counts present' if ok else 'MISSING (see the exit codes above)'}")
if PART == "B":
    f = os.path.join("reports/m9/full", TAG, "ablation", "confusions.csv")
    print("m9 ablation:", "per-client counts present" if os.path.exists(f) and "csv_client" in open(f).read() else "MISSING")
print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout[:3000])
